# 12.4 截取時間框時，為什麼要讓兩端變小？


上一節取出的 400 點不一定剛好在波形相同的地方起止。頻率計算若把這一框當作可重複的一段，首尾接起來可能突然跳一下。頻率計算也會把這個接點跳變算進去，帶出原本不是單音本身的額外頻率成分，讓能量分散。

可以讓首尾數字逐漸接近零、中間保留較大比重。這排乘數叫窗函數；本例用 Hann 窗，每個波形樣本乘所在位置的權重。


In [1]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [2]:
import torch
from tiny_perceptron.multimodal import tone

frame = tone()[:400]
window = torch.hann_window(400)
weighted = frame * window
print("起點權重", window[0].item(), "中間權重", window[200].item())
print("最後權重", round(window[-1].item(), 6))
print("原能量", round(frame.square().sum().item(), 2))
print("加窗能量", round(weighted.square().sum().item(), 2))

起點權重 0.0 中間權重 1.0
最後權重 6.2e-05
原能量 50.0
加窗能量 18.75


起點權重為 0，中間約 1，最後約 0.000062。PyTorch 預設採供週期延續使用的版本，所以最後一點很接近零卻不是正好零。若改用包含兩端的對稱版本，公式與最後一點會稍有不同。

原能量是振幅平方後相加，約 50；乘窗後約 18.75。這只是同一框經過加權，並沒有模型學習，也不表示音源變小了。加窗緩和的是首尾跳變帶出的額外頻率成分，並不表示所有能量都會集中成一點。另一個代價是主要頻率附近的峰本身會變寬：若把各頻率的能量畫成高低，主峰是主要頻率附近最高的一帶；這一帶變寬，兩個較近頻率的峰就可能更難分開。

窗長 400、取樣率 16000 仍表示 25 毫秒，乘窗不更動時間點。後續若要比較不同錄音的能量，必須使用一致的窗與正規化約定，不能把不同配方的數字直接當成同一尺度。

練習將 400 個權重都設成 1：weighted 應等於 frame、兩個能量相同。再說明這個控制保留了振幅，卻沒有緩和首尾接合的不連續。

<details>
<summary>回顧與查證</summary>

可回顧：[12.3時間框](https://birdhackor.github.io/tiny-perceptron-vlm/12.3.html)。

</details>
